In [ ]:
import re
import pandas as pd
df = pd.read_csv("/content/social_media_data.csv")
# Remove rows where user_name is exactly "Mr. Tweet"
df = df[df["user_name"] != "Mr. Tweet"]

In [ ]:
summary = df["user_name"].value_counts().reset_index()
summary.columns = ["Person", "Number of Posts"]

print(summary)

         Person  Number of Posts
0  Donald Trump            38141
1     Joe Biden             8648
2     Elon Musk             5825


In [ ]:
import re
import pandas as pd

# ===== 1) Load dataset =====
df = pd.read_csv("/content/social_media_data.csv")

text_col = "text"

# ===== 2) Categorized high-impact keywords =====
KEYWORDS = {
    "geopolitical_trade": [
        "sanction","sanctions","tariff","tariffs","trade war","embargo",
        "export ban","import ban","china","russia","ukraine","taiwan",
        "nato","strike","attack","war","conflict","invasion",
        "ceasefire","blockade","missile","airstrike"
    ],
    "monetary_fed_inflation": [
        "federal reserve","fed","fomc","powell",
        "rate hike","interest rate hike","rate cut","interest rate cut",
        "tightening","easing","quantitative tightening","quantitative easing",
        "balance sheet","inflation","cpi","ppi","core inflation",
        "recession","soft landing","hard landing",
        "unemployment","jobs report","nonfarm payrolls","payrolls"
    ],
    "fiscal_regulatory": [
        "stimulus","tax cut","tax cuts","tax hike","tax hikes","corporate tax",
        "regulation","regulations","deregulation","antitrust",
        "subsidy","subsidies","bailout","bailouts",
        "debt ceiling","government shutdown","shutdown",
        "budget deal","spending bill","executive order","price cap"
    ],
    "corporate_events": [
        "earnings miss","earnings beat","guidance cut","guidance raised",
        "bankruptcy","default","downgrade","rating cut",
        "ipo","public offering","acquisition","merger","takeover",
        "lawsuit","investigation","probe","recall","data breach"
    ],
    "financial_stress": [
        "bank failure","liquidity crisis","credit crisis","collapse",
        "contagion","systemic risk","vix","panic","bank run"
    ],
    "energy_commodities": [
        "oil embargo","oil shock","opec","production cut",
        "gas prices","energy crisis","crude","brent","wti","natural gas"
    ],
    "crypto": [
        "bitcoin","btc","crypto","cryptocurrency","ethereum","eth",
        "dogecoin","crypto crash","spot etf","etf approval"
    ],
    "urgent_language": [
        "emergency","crisis","immediately",
        "effective immediately","breaking",
        "major announcement","urgent","alert"
    ]
}
KEYWORDS["macro_data"] = [
    "cpi","ppi","pce","core pce","jobs report","payrolls","nonfarm payrolls","unemployment",
    "gdp","pmi","ism","retail sales","consumer confidence","inflation data"
]

KEYWORDS["rates_credit"] = [
    "treasury","10-year","2-year","yield","yields","yield curve","inversion",
    "credit spread","spreads","default","downgrade","rating cut"
]

KEYWORDS["market_shock"] = [
    "selloff","sell-off","crash","plunge","tumble","tank","collapse","panic",
    "spike","surge","record high","record low","circuit breaker","halted","trading halt"
]

# ===== 3) Compile regex patterns =====
def compile_patterns(kws):
    patterns = []
    for kw in kws:
        kw_l = kw.lower()
        if re.fullmatch(r"[a-z]+", kw_l):
            patterns.append(re.compile(rf"\b{re.escape(kw_l)}\b", re.IGNORECASE))
        else:
            patterns.append(re.compile(re.escape(kw_l), re.IGNORECASE))
    return patterns

PATTERNS = {cat: compile_patterns(kws) for cat, kws in KEYWORDS.items()}

# ===== 4) Label posts =====
def label_text(text):
    if pd.isna(text):
        return {cat: [] for cat in KEYWORDS}
    t = str(text).lower()
    out = {}
    for cat, kws in KEYWORDS.items():
        matches = []
        for kw, pat in zip(kws, PATTERNS[cat]):
            if pat.search(t):
                matches.append(kw)
        out[cat] = matches
    return out

hits = df[text_col].apply(label_text)

for cat in KEYWORDS:
    df[f"kw_{cat}_hits"] = hits.apply(lambda d, c=cat: d[c])
    df[f"kw_{cat}_flag"] = df[f"kw_{cat}_hits"].apply(lambda x: int(len(x) > 0))

df["kw_total_hits"] = df[[f"kw_{cat}_hits" for cat in KEYWORDS]].apply(
    lambda row: sum(len(x) for x in row), axis=1
)

df["kw_any_flag"] = (df["kw_total_hits"] > 0).astype(int)

# ===== 5) Summary =====
print("\nCategory counts:")
print(df[[f"kw_{cat}_flag" for cat in KEYWORDS]].sum().sort_values(ascending=False))

print("\nPosts with ANY high-impact keywords:",
      df["kw_any_flag"].sum(), "out of", len(df))

# ===== 6) Save =====
df.to_csv("/content/social_media_data_labeled.csv", index=False)
print("\nSaved to /content/social_media_data_labeled.csv")

# Preview flagged examples
df[df["kw_any_flag"] == 1][[text_col, "kw_total_hits"]].head(10)


Category counts:
kw_geopolitical_trade_flag        2038
kw_urgent_language_flag            912
kw_fiscal_regulatory_flag          549
kw_monetary_fed_inflation_flag     491
kw_corporate_events_flag           329
kw_macro_data_flag                 301
kw_market_shock_flag               205
kw_rates_credit_flag                43
kw_energy_commodities_flag          39
kw_financial_stress_flag            36
kw_crypto_flag                       7
dtype: int64

Posts with ANY high-impact keywords: 4365 out of 52693

Saved to /content/social_media_data_labeled.csv


,text,kw_total_hits
25,trump himself never ever filed for bankruptcy ...,1
28,icahn kravis buffet amp many other top busines...,1
431,my condolences and prayers to the victims of t...,1
489,via by donald trump francestrict gun laws enab...,1
665,despite what the haters and losers like to say...,1
666,i say donald trump takeover the whitehouse now...,1
674,well haspoint but caesar never filed for bankr...,1
702,got out of class and immediately turned donald...,1
920,word is that is firing sleepy eyes chuck todd ...,1
1089,via by doral mayor declares emergency to give ...,1


In [ ]:
PERSON_KEYWORDS = {
    "Joe Biden": {
        "biden_policy_macro": [
            "executive order", "white house", "press briefing", "state of the union",
            "budget", "spending bill", "infrastructure", "chips act", "chips", "semiconductor",
            "student loans", "debt relief", "tax credit", "irs", "regulation",
            "ukraine", "russia", "sanctions", "china", "tariffs",
            "jobs", "unemployment", "payrolls", "inflation", "cpi", "gas prices"
        ]
    },
    "Elon Musk": {
        "musk_markets_tech": [
            "tesla", "tsla", "spacex", "starlink", "x", "twitter",
            "ai", "grok", "robotaxi", "fsd", "autopilot",
            "earnings", "guidance", "deliveries", "margin", "revenue",
            "chip", "chips", "nvidia", "nvda", "apple", "aapl", "google", "msft",
            "rate hike", "rate cut", "fed", "inflation", "cpi"
        ]
    }
}
import re
import pandas as pd

text_col = "text"
person_col = "user_name"

def compile_kw_list(words):
    pats = []
    for kw in words:
        kw_l = kw.lower()
        if re.fullmatch(r"[a-z]+", kw_l):
            pats.append(re.compile(rf"\b{re.escape(kw_l)}\b", re.IGNORECASE))
        else:
            pats.append(re.compile(re.escape(kw_l), re.IGNORECASE))
    return pats

# Compile supplemental patterns
PERSON_PATTERNS = {}
for person, cat_map in PERSON_KEYWORDS.items():
    PERSON_PATTERNS[person] = {cat: compile_kw_list(kws) for cat, kws in cat_map.items()}

def count_matches(text, patterns):
    if pd.isna(text):
        return 0
    t = str(text).lower()
    return sum(1 for pat in patterns if pat.search(t))

# Make sure core kw_total_hits already exists from your earlier KEYWORDS run.
# If not, rerun your existing keyword labeling first.

# Compute supplement hits
df["supp_hits"] = 0

for person, cat_map in PERSON_PATTERNS.items():
    mask = df[person_col] == person
    if mask.any():
        # sum across that person's supplemental categories
        total = 0
        for cat, pats in cat_map.items():
            df.loc[mask, f"supp_{cat}_hits"] = df.loc[mask, text_col].apply(lambda x: count_matches(x, pats))
            total += df.loc[mask, f"supp_{cat}_hits"]
        df.loc[mask, "supp_hits"] = total


In [ ]:
df["combined_hits"] = df["kw_total_hits"] + df["supp_hits"]

In [ ]:
def keep_row(r):
    name = r[person_col]
    if name == "Donald Trump":
        return r["kw_total_hits"] >= 2
    elif name in ["Joe Biden", "Elon Musk"]:
        return r["combined_hits"] >= 1
    else:
        return r["kw_total_hits"] >= 5

df_balanced = df[df.apply(keep_row, axis=1)].copy()
df_balanced[person_col].value_counts()

,count
user_name,
Joe Biden,1194
Donald Trump,800
Elon Musk,784


In [ ]:
tmp = df_balanced.copy()

tmp["dt0"] = pd.to_datetime(tmp["datetime"], errors="coerce")

if tmp["dt0"].dt.tz is None:
    tmp["dt0"] = tmp["dt0"].dt.tz_localize(
        "America/New_York",
        ambiguous="NaT",
        nonexistent="shift_forward"
    )

dt_local = tmp["dt0"].dt.tz_convert("America/New_York")

rth_mask = (
    ((dt_local.dt.hour > 9) | ((dt_local.dt.hour == 9) & (dt_local.dt.minute >= 30))) &
    (dt_local.dt.hour < 16)
)

tmp[rth_mask]["user_name"].value_counts()

,count
user_name,
Donald Trump,413
Joe Biden,162
Elon Musk,106


In [ ]:
sentiment_col = "avg_sentiment"
tmp[rth_mask].groupby(person_col)[sentiment_col].agg(mean="mean", std="std", n="count")

,mean,std,n
user_name,,,
Donald Trump,-0.047639,0.490353,413
Elon Musk,0.014539,0.324824,106
Joe Biden,-0.052048,0.415520,162


In [ ]:
events = tmp[rth_mask].copy()

print("New events size:", len(events))
events["user_name"].value_counts()

New events size: 681


,count
user_name,
Donald Trump,413
Joe Biden,162
Elon Musk,106


In [ ]:
dt_local = events["dt0"].dt.tz_convert("America/New_York")

#close_mask = dt_local.dt.time <= pd.to_datetime("15:30:00").time()
close_mask = dt_local.dt.time >= pd.to_datetime("10:00:00").time()
events = events[close_mask].copy()

print("After removing near-close events:", len(events))
events["user_name"].value_counts()

After removing near-close events: 600


,count
user_name,
Donald Trump,391
Joe Biden,128
Elon Musk,81


In [ ]:
import numpy as np

# Separate Trump and non-Trump
trump = events[events["user_name"] == "Donald Trump"]
non_trump = events[events["user_name"] != "Donald Trump"]

# Randomly sample 150 Trump posts
trump_sampled = trump.sample(n=150, random_state=42)

# Combine back
events = pd.concat([trump_sampled, non_trump]).copy()

# Optional: shuffle rows
events = events.sample(frac=1, random_state=42).reset_index(drop=True)

# Check counts
events["user_name"].value_counts()

,count
user_name,
Donald Trump,150
Joe Biden,128
Elon Musk,81


In [ ]:
import pandas as pd

# Start from whichever set you want
events = events.copy()
# 1) Duplicate each post into two rows: SPY + QQQ
events = events.assign(symbol=[["SPY", "QQQ"]] * len(events)).explode("symbol").reset_index(drop=True)

# 2) Parse timestamps
time_col = "datetime"
events[time_col] = pd.to_datetime(events[time_col], errors="coerce")
events = events.dropna(subset=[time_col, "symbol"])

# Ensure timezone is US/Eastern (adjust if your datetimes are already tz-aware)
if events[time_col].dt.tz is None:
    events[time_col] = events[time_col].dt.tz_localize("America/New_York")

# 3) Build per-event windows
events["date_et"] = events[time_col].dt.tz_convert("America/New_York").dt.date
events["win_start"] = events[time_col] - pd.Timedelta(minutes=30)
events["win_end"]   = events[time_col] + pd.Timedelta(minutes=30)

# 4) Collapse to one window per (symbol, date)
symbol_day_windows = (
    events.groupby(["symbol", "date_et"], as_index=False)
          .agg(win_start=("win_start", "min"),
               win_end=("win_end", "max"),
               n_events=("date_et", "size"))
          .sort_values(["date_et", "symbol"])
)

print("Events after SPY+QQQ expansion:", len(events))
print("Unique symbol-days (bulk queries):", len(symbol_day_windows))

symbol_day_windows.head()

Events after SPY+QQQ expansion: 718
Unique symbol-days (bulk queries): 562


,symbol,date_et,win_start,win_end,n_events
0,QQQ,2012-04-14,2012-04-14 14:57:00-04:00,2012-04-14 15:57:00-04:00,1
281,SPY,2012-04-14,2012-04-14 14:57:00-04:00,2012-04-14 15:57:00-04:00,1
1,QQQ,2012-08-22,2012-08-22 14:31:00-04:00,2012-08-22 15:31:00-04:00,1
282,SPY,2012-08-22,2012-08-22 14:31:00-04:00,2012-08-22 15:31:00-04:00,1
2,QQQ,2012-09-18,2012-09-18 12:40:00-04:00,2012-09-18 14:40:00-04:00,5


In [ ]:
symbol_day_windows.to_csv("/content/symbol_day_windows_spy_qqq.csv", index=False)
print("Saved: /content/symbol_day_windows_spy_qqq.csv")

Saved: /content/symbol_day_windows_spy_qqq.csv


In [ ]:
# One window per date (covers BOTH SPY+QQQ on that day)
date_windows = (
    events.groupby("date_et", as_index=False)
          .agg(win_start=("win_start", "min"),
               win_end=("win_end", "max"),
               n_events=("date_et", "size"))
          .sort_values("date_et")
)

print("Unique dates (queries if 1 per date):", len(date_windows))
date_windows.head()

Unique dates (queries if 1 per date): 281


,date_et,win_start,win_end,n_events
0,2012-04-14,2012-04-14 14:57:00-04:00,2012-04-14 15:57:00-04:00,2
1,2012-08-22,2012-08-22 14:31:00-04:00,2012-08-22 15:31:00-04:00,2
2,2012-09-18,2012-09-18 12:40:00-04:00,2012-09-18 14:40:00-04:00,10
3,2014-05-14,2014-05-14 14:40:00-04:00,2014-05-14 15:40:00-04:00,2
4,2015-10-13,2015-10-13 11:09:52-04:00,2015-10-13 12:09:52-04:00,2


In [ ]:
import pandas as pd

def ms_since_midnight(ts):
    ts = ts.tz_convert("America/New_York")
    return (ts.hour*3600 + ts.minute*60 + ts.second)*1000 + ts.microsecond//1000

date_windows["date"] = pd.to_datetime(date_windows["date_et"])
date_windows["start_ms"] = date_windows["win_start"].apply(ms_since_midnight).astype(int)
date_windows["end_ms"]   = date_windows["win_end"].apply(ms_since_midnight).astype(int)

date_windows[["date_et","start_ms","end_ms","n_events"]].tail()

,date_et,start_ms,end_ms,n_events
276,2023-04-25,46477000,56501000,4
277,2023-04-29,47921000,51521000,2
278,2023-05-10,50259000,53859000,2
279,2023-06-03,41725000,45325000,2
280,2023-06-06,48052000,51652000,2


In [ ]:
!pip install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 101.3 MB/s eta 0:00:00


In [ ]:
import wrds
db = wrds.Connection()

# See TAQ millisecond tables available in your WRDS
db.list_tables(library="taqmsec")[:20]

Enter your WRDS username [root]:mm7252
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


['complete_nbbo_2003',
 'complete_nbbo_20030910',
 'complete_nbbo_20030911',
 'complete_nbbo_20030912',
 'complete_nbbo_20030915',
 'complete_nbbo_20030916',
 'complete_nbbo_20030917',
 'complete_nbbo_20030918',
 'complete_nbbo_20030919',
 'complete_nbbo_20030922',
 'complete_nbbo_20030923',
 'complete_nbbo_20030924',
 'complete_nbbo_20030925',
 'complete_nbbo_20030926',
 'complete_nbbo_20030929',
 'complete_nbbo_20030930',
 'complete_nbbo_20031001',
 'complete_nbbo_20031002',
 'complete_nbbo_20031003',
 'complete_nbbo_20031006']

In [ ]:
tables = db.list_tables(library="taqmsec")

# Look for anything that looks like trades
[t for t in tables if "ctm" in t.lower() or "trade" in t.lower()][:50]

['ctm_2003',
 'ctm_20030910',
 'ctm_20030911',
 'ctm_20030912',
 'ctm_20030915',
 'ctm_20030916',
 'ctm_20030917',
 'ctm_20030918',
 'ctm_20030919',
 'ctm_20030922',
 'ctm_20030923',
 'ctm_20030924',
 'ctm_20030925',
 'ctm_20030926',
 'ctm_20030929',
 'ctm_20030930',
 'ctm_20031001',
 'ctm_20031002',
 'ctm_20031003',
 'ctm_20031006',
 'ctm_20031007',
 'ctm_20031008',
 'ctm_20031009',
 'ctm_20031010',
 'ctm_20031013',
 'ctm_20031014',
 'ctm_20031015',
 'ctm_20031016',
 'ctm_20031017',
 'ctm_20031020',
 'ctm_20031021',
 'ctm_20031022',
 'ctm_20031023',
 'ctm_20031024',
 'ctm_20031027',
 'ctm_20031028',
 'ctm_20031029',
 'ctm_20031030',
 'ctm_20031031',
 'ctm_20031103',
 'ctm_20031104',
 'ctm_20031105',
 'ctm_20031106',
 'ctm_20031107',
 'ctm_20031110',
 'ctm_20031111',
 'ctm_20031112',
 'ctm_20031113',
 'ctm_20031114',
 'ctm_20031117']

In [ ]:
libs = db.list_libraries()
[l for l in libs if "taq" in l.lower()]

['contrib_liquidity_taq',
 'taqm_2003',
 'taqm_2004',
 'taqm_2005',
 'taqm_2006',
 'taqm_2007',
 'taqm_2008',
 'taqm_2009',
 'taqm_2010',
 'taqm_2011',
 'taqm_2012',
 'taqm_2013',
 'taqm_2014',
 'taqm_2015',
 'taqm_2016',
 'taqm_2017',
 'taqm_2018',
 'taqm_2019',
 'taqm_2020',
 'taqm_2021',
 'taqm_2022',
 'taqm_2023',
 'taqm_2024',
 'taqm_2025',
 'taqm_2026',
 'taqmsamp',
 'taqmsamp_all',
 'taqmsec',
 'taqsamp',
 'taqsamp_all',
 'wrdsapps_link_crsp_taq',
 'wrdsapps_link_crsp_taqm']

In [ ]:
##db.list_tables(library="taq")[:50]
db.list_tables(library="taqm_2023")[:50]      # if it exists
##db.list_tables(library="taqms")[:50]     # if it exists

['complete_nbbo_2023',
 'complete_nbbo_20230103',
 'complete_nbbo_20230104',
 'complete_nbbo_20230105',
 'complete_nbbo_20230106',
 'complete_nbbo_20230109',
 'complete_nbbo_20230110',
 'complete_nbbo_20230111',
 'complete_nbbo_20230112',
 'complete_nbbo_20230113',
 'complete_nbbo_20230117',
 'complete_nbbo_20230118',
 'complete_nbbo_20230119',
 'complete_nbbo_20230120',
 'complete_nbbo_20230123',
 'complete_nbbo_20230124',
 'complete_nbbo_20230125',
 'complete_nbbo_20230126',
 'complete_nbbo_20230127',
 'complete_nbbo_20230130',
 'complete_nbbo_20230131',
 'complete_nbbo_20230201',
 'complete_nbbo_20230202',
 'complete_nbbo_20230203',
 'complete_nbbo_20230206',
 'complete_nbbo_20230207',
 'complete_nbbo_20230208',
 'complete_nbbo_20230209',
 'complete_nbbo_20230210',
 'complete_nbbo_20230213',
 'complete_nbbo_20230214',
 'complete_nbbo_20230215',
 'complete_nbbo_20230216',
 'complete_nbbo_20230217',
 'complete_nbbo_20230221',
 'complete_nbbo_20230222',
 'complete_nbbo_20230223',
 'com

In [ ]:
year_lib = "taqm_2023"
tables = db.list_tables(library=year_lib)

print("CTM:", [t for t in tables if "ctm" in t.lower()][:20])
print("TRADES:", [t for t in tables if "trade" in t.lower()][:20])
print("NON-NBBO sample:", [t for t in tables if "nbbo" not in t.lower()][:50])

CTM: ['ctm_2023', 'ctm_20230103', 'ctm_20230104', 'ctm_20230105', 'ctm_20230106', 'ctm_20230109', 'ctm_20230110', 'ctm_20230111', 'ctm_20230112', 'ctm_20230113', 'ctm_20230117', 'ctm_20230118', 'ctm_20230119', 'ctm_20230120', 'ctm_20230123', 'ctm_20230124', 'ctm_20230125', 'ctm_20230126', 'ctm_20230127', 'ctm_20230130']
TRADES: []
NON-NBBO sample: ['cqm_2023', 'cqm_20230103', 'cqm_20230104', 'cqm_20230105', 'cqm_20230106', 'cqm_20230109', 'cqm_20230110', 'cqm_20230111', 'cqm_20230112', 'cqm_20230113', 'cqm_20230117', 'cqm_20230118', 'cqm_20230119', 'cqm_20230120', 'cqm_20230123', 'cqm_20230124', 'cqm_20230125', 'cqm_20230126', 'cqm_20230127', 'cqm_20230130', 'cqm_20230131', 'cqm_20230201', 'cqm_20230202', 'cqm_20230203', 'cqm_20230206', 'cqm_20230207', 'cqm_20230208', 'cqm_20230209', 'cqm_20230210', 'cqm_20230213', 'cqm_20230214', 'cqm_20230215', 'cqm_20230216', 'cqm_20230217', 'cqm_20230221', 'cqm_20230222', 'cqm_20230223', 'cqm_20230224', 'cqm_20230227', 'cqm_20230228', 'cqm_20230301

In [ ]:
db.describe_table(library="taqm_2023", table="ctm_2023")

Approximately 17816449457 rows in taqm_2023.ctm_2023.


,name,nullable,type,comment
0,date,False,DATE,Date of trade
1,time_m,False,TIME,Time to the microsecond
2,time_m_nano,True,SMALLINT,Nanosecond offset
3,ex,True,VARCHAR(1),Exchange that issued the trade
4,sym_root,False,TEXT,Security symbol root
5,sym_suffix,True,TEXT,Security symbol suffix
6,tr_scond,True,VARCHAR(4),Trade Sale Condition (up to 4 codes)
7,size,True,INTEGER,Volume of trade
8,price,True,NUMERIC,Price of trade
9,tr_stop_ind,True,VARCHAR(1),Trade Stop Stock Indicator (NYSE Only)


In [ ]:
import pandas as pd

# events should already be your SPY+QQQ exploded dataset
time_col = "datetime"
events[time_col] = pd.to_datetime(events[time_col], errors="coerce")
events = events.dropna(subset=[time_col])

if events[time_col].dt.tz is None:
    events[time_col] = events[time_col].dt.tz_localize("America/New_York")

events["date_et"] = events[time_col].dt.tz_convert("America/New_York").dt.date
events["win_start"] = events[time_col] - pd.Timedelta(minutes=30)
events["win_end"]   = events[time_col] + pd.Timedelta(minutes=30)

date_windows = (
    events.groupby("date_et", as_index=False)
          .agg(win_start=("win_start","min"),
               win_end=("win_end","max"),
               n_events=("date_et","size"))
          .sort_values("date_et")
)

# Convert to TIME strings (microseconds supported, but seconds are fine)
date_windows["start_time"] = date_windows["win_start"].dt.tz_convert("America/New_York").dt.strftime("%H:%M:%S")
date_windows["end_time"]   = date_windows["win_end"].dt.tz_convert("America/New_York").dt.strftime("%H:%M:%S")

print("Unique dates to query:", len(date_windows))
date_windows.head()

Unique dates to query: 281


,date_et,win_start,win_end,n_events,start_time,end_time
0,2012-04-14,2012-04-14 14:57:00-04:00,2012-04-14 15:57:00-04:00,2,14:57:00,15:57:00
1,2012-08-22,2012-08-22 14:31:00-04:00,2012-08-22 15:31:00-04:00,2,14:31:00,15:31:00
2,2012-09-18,2012-09-18 12:40:00-04:00,2012-09-18 14:40:00-04:00,10,12:40:00,14:40:00
3,2014-05-14,2014-05-14 14:40:00-04:00,2014-05-14 15:40:00-04:00,2,14:40:00,15:40:00
4,2015-10-13,2015-10-13 11:09:52-04:00,2015-10-13 12:09:52-04:00,2,11:09:52,12:09:52


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def get_tables_for_year(year):
    lib = f"taqm_{year}"
    return set(db.list_tables(library=lib))

In [ ]:
try:
    db.connection.rollback()
except Exception as e:
    print("Rollback failed:", e)

In [ ]:
import pandas as pd

def _to_time_str(t):
    # t can be 'HH:MM:SS' or datetime.time
    return pd.to_datetime(str(t)).strftime("%H:%M:%S")

def extend_window(start_time, end_time, minutes=30,
                  floor="09:30:00", ceil="16:00:00"):
    s = pd.to_datetime(_to_time_str(start_time))
    e = pd.to_datetime(_to_time_str(end_time))
    s_ext = s - pd.Timedelta(minutes=minutes)
    e_ext = e + pd.Timedelta(minutes=minutes)

    # clamp (optional but recommended)
    s_floor = pd.to_datetime(floor)
    e_ceil  = pd.to_datetime(ceil)
    s_ext = max(s_ext, s_floor)
    e_ext = min(e_ext, e_ceil)

    return s_ext.strftime("%H:%M:%S"), e_ext.strftime("%H:%M:%S")


symbols_sql = "('SPY','QQQ')"
all_chunks = []

for _, r in date_windows.iterrows():
    d = pd.to_datetime(r["date_et"])
    y = d.year
    ymd = d.strftime("%Y%m%d")

    lib = f"taqm_{y}"
    daily_table = f"ctm_{ymd}"
    yearly_table = f"ctm_{y}"

    # ORIGINAL
    start_t = r["start_time"]
    end_t = r["end_time"]

    # NEW: extend by ±30 minutes
    start_t_ext, end_t_ext = extend_window(start_t, end_t, minutes=30)

    d_str = d.strftime("%Y-%m-%d")
    tables = get_tables_for_year(y)

    if daily_table in tables:
        sql = f"""
            select date, time_m, sym_root, sym_suffix, price, size
            from {lib}.{daily_table}
            where sym_root in {symbols_sql}
              and time_m between '{start_t_ext}'::time and '{end_t_ext}'::time
        """
    else:
        sql = f"""
            select date, time_m, sym_root, sym_suffix, price, size
            from {lib}.{yearly_table}
            where date = '{d_str}'
              and sym_root in {symbols_sql}
              and time_m between '{start_t_ext}'::time and '{end_t_ext}'::time
        """

    chunk = db.raw_sql(sql)
    chunk["date_et"] = d.date()
    all_chunks.append(chunk)

trades = pd.concat(all_chunks, ignore_index=True)
print("Total trade rows pulled:", len(trades))
print(trades["sym_root"].value_counts())

/tmp/ipython-input-6598/2137857172.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  trades = pd.concat(all_chunks, ignore_index=True)


Total trade rows pulled: 46409339
sym_root
SPY    30741546
QQQ    15667793
Name: count, dtype: Int64


In [ ]:
# build a proper ET timestamp
trades["ts"] = pd.to_datetime(trades["date"].astype(str) + " " + trades["time_m"].astype(str))
trades = trades.sort_values(["sym_root", "ts"])

# bar close = last trade in each bin
bars_5 = (trades
          .set_index("ts")
          .groupby("sym_root")["price"]
          .resample("5min")
          .last()
          .dropna()
          .reset_index()
          .rename(columns={"price": "close_5"}))

bars_30 = (trades
           .set_index("ts")
           .groupby("sym_root")["price"]
           .resample("30min")
           .last()
           .dropna()
           .reset_index()
           .rename(columns={"price": "close_30"}))

ValueError: time data "2012-08-22 14:02:48" doesn't match format "%Y-%m-%d %H:%M:%S.%f", at position 597. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
trades = trades[["date", "time_m", "sym_root", "price"]].copy()

In [ ]:
import pandas as pd

trades = trades.copy()

# Ensure date is datetime (midnight)
d0 = pd.to_datetime(trades["date"])

# Convert time_m (datetime.time) into a timedelta since midnight
# This handles both second-only and microsecond times safely.
t = pd.to_timedelta(trades["time_m"].astype(str))

trades["dt"] = d0 + t
trades["dt"] = trades["dt"].dt.tz_localize("America/New_York")

trades = trades.sort_values(["sym_root", "dt"])

In [ ]:
trades["dt"].head(), trades["dt"].tail()

(0   2012-08-22 14:01:00.182000-04:00
 1   2012-08-22 14:01:00.183000-04:00
 2   2012-08-22 14:01:00.351000-04:00
 3   2012-08-22 14:01:00.398000-04:00
 4   2012-08-22 14:01:00.398000-04:00
 Name: dt, dtype: datetime64[ns, America/New_York],
 46409334   2023-06-06 14:50:51.320520-04:00
 46409335   2023-06-06 14:50:51.359865-04:00
 46409336   2023-06-06 14:50:51.426639-04:00
 46409337   2023-06-06 14:50:51.446535-04:00
 46409338   2023-06-06 14:50:51.737072-04:00
 Name: dt, dtype: datetime64[ns, America/New_York])

In [ ]:
# Keep only what you need (saves RAM)
t = trades.rename(columns={"sym_root": "symbol"})[["symbol", "dt", "price"]].copy()
t = t.sort_values(["symbol", "dt"])

In [ ]:
events = events.copy()

events["dt0"] = pd.to_datetime(events["datetime"], errors="coerce")
if events["dt0"].dt.tz is None:
    events["dt0"] = events["dt0"].dt.tz_localize("America/New_York")

events = events.dropna(subset=["dt0", "symbol"]).sort_values(["symbol", "dt0"])

events["dt5"]  = events["dt0"] + pd.Timedelta(minutes=5)
events["dt30"] = events["dt0"] + pd.Timedelta(minutes=30)
events["dt_5"]  = events["dt0"] - pd.Timedelta(minutes=5)
events["dt_30"] = events["dt0"] - pd.Timedelta(minutes=30)

In [ ]:
events["symbol"] = events["symbol"].astype("object")
t["symbol"] = t["symbol"].astype("object")

# and re-sort (merge_asof requires sorted)
events = events.sort_values(["symbol", "dt0"])
t = t.sort_values(["symbol", "dt"])

In [ ]:
def match_price_forward(events_df, target_col):
    left = (
        events_df[["symbol", target_col]]
        .rename(columns={target_col: "dt"})
        .dropna(subset=["symbol", "dt"])
        .sort_values(["symbol", "dt"], kind="mergesort")  # stable sort
        .reset_index(drop=True)
    )

    right = (
        t.dropna(subset=["symbol", "dt"])
         .sort_values(["symbol", "dt"], kind="mergesort")
         .reset_index(drop=True)
    )

    matched = pd.merge_asof(
        left, right,
        by="symbol",
        on="dt",
        direction="forward",
        allow_exact_matches=True
    )
    return matched["price"].to_numpy()

In [ ]:
tmp = events[["symbol","dt0"]].rename(columns={"dt0":"dt"}).sort_values(["symbol","dt"], kind="mergesort")
print(tmp["dt"].is_monotonic_increasing)  # will be False because symbols interleave

False


In [ ]:
import numpy as np
import pandas as pd

# Ensure clean types
events["symbol"] = events["symbol"].astype(str)
t["symbol"] = t["symbol"].astype(str)

def match_symbol_forward(events_sym, trades_sym, target_col):
    left = (
        events_sym[[target_col]]
        .rename(columns={target_col: "dt"})
        .dropna(subset=["dt"])
        .sort_values("dt")
        .reset_index()
    )

    right = (
        trades_sym[["dt","price"]]
        .dropna(subset=["dt"])
        .sort_values("dt")
        .reset_index(drop=True)
    )

    matched = pd.merge_asof(
        left, right,
        on="dt",
        direction="forward",
        allow_exact_matches=True
    )

    # restore original order
    out = matched.set_index("index")["price"].reindex(events_sym.index)
    return out

def add_prices(events, t, target_col, out_col):
    out = pd.Series(index=events.index, dtype="float64")

    for sym in ["SPY", "QQQ"]:
        ev = events[events["symbol"] == sym]
        tr = t[t["symbol"] == sym]
        out.loc[ev.index] = match_symbol_forward(ev, tr, target_col)

    events[out_col] = out
    return events

# Run matching
events = add_prices(events, t, "dt0", "p0")
events = add_prices(events, t, "dt5", "p5")
events = add_prices(events, t, "dt30", "p30")
events = add_prices(events, t, "dt_5", "p_5")
events = add_prices(events, t, "dt_30", "p_30")

In [ ]:
events["sym_ret_5m"]  = events["p5"]  / events["p_5"] - 1
events["sym_ret_30m"] = events["p30"] / events["p_30"] - 1
events["ret_5m"]  = events["p5"]  / events["p0"] - 1
events["ret_30m"] = events["p30"] / events["p0"] - 1
events["pre_ret_5m"]  = events["p0"]  / events["p_5"] - 1
events["pre_ret_30m"] = events["p0"] / events["p_30"] - 1

In [ ]:
print("Missing p0/p5/p30 rates:")
print(events[["p0","p5","p30", "p_5", "p_30"]].isna().mean())

print(events[["ret_5m","ret_30m", "pre_ret_5m", "pre_ret_30m", "sym_ret_5m", "sym_ret_30m"]].describe())
events[["user_name","symbol","dt0","p0","p5","p30","ret_5m","ret_30m"]].head(10)

Missing p0/p5/p30 rates:
p0      0.0
p5      0.0
p30     0.0
p_5     0.0
p_30    0.0
dtype: float64
           ret_5m     ret_30m  pre_ret_5m  pre_ret_30m  sym_ret_5m  \
count  718.000000  718.000000  718.000000   718.000000  718.000000   
mean     0.000053    0.000379    0.000033    -0.000102    0.000087   
std      0.001093    0.009987    0.001269     0.002379    0.001716   
min     -0.010641   -0.059060   -0.007583    -0.015658   -0.017686   
25%     -0.000109   -0.000608   -0.000238    -0.000534   -0.000219   
50%      0.000000    0.000000    0.000000     0.000000    0.000000   
75%      0.000361    0.000698    0.000266     0.000558    0.000510   
max      0.007403    0.179656    0.013538     0.012227    0.014467   

       sym_ret_30m  
count   718.000000  
mean      0.000276  
std       0.010213  
min      -0.061947  
25%      -0.000682  
50%       0.000000  
75%       0.001167  
max       0.179228  


,user_name,symbol,dt0,p0,p5,p30,ret_5m,ret_30m
149,Joe Biden,QQQ,2012-04-14 15:27:00-04:00,68.2400,68.240,68.2400,0.000000,0.000000
91,Joe Biden,QQQ,2012-08-22 15:01:00-04:00,68.4500,68.450,68.4400,0.000000,-0.000146
367,Joe Biden,QQQ,2012-09-18 13:10:00-04:00,70.1400,70.184,70.1800,0.000627,0.000570
359,Joe Biden,QQQ,2012-09-18 13:25:00-04:00,70.1950,70.190,70.1500,-0.000071,-0.000641
667,Joe Biden,QQQ,2012-09-18 13:35:00-04:00,70.1845,70.180,70.1300,-0.000064,-0.000777
577,Joe Biden,QQQ,2012-09-18 13:45:00-04:00,70.1800,70.160,70.1800,-0.000285,0.000000
103,Joe Biden,QQQ,2012-09-18 14:10:00-04:00,70.1699,70.180,70.1400,0.000144,-0.000426
249,Joe Biden,QQQ,2014-05-14 15:10:00-04:00,87.8000,87.840,87.7100,0.000456,-0.001025
705,Donald Trump,QQQ,2015-10-13 11:39:52-04:00,106.9810,106.920,106.9100,-0.000570,-0.000664
411,Donald Trump,QQQ,2015-12-01 15:20:40-05:00,114.8690,114.900,115.0625,0.000270,0.001685


In [ ]:
cols = ["user_name", "person", "symbol", "dt0", "p0", "dt5", "p5", "dt30", "p30", "ret_5m", "ret_30m", "text"]

# In case your person column is user_name only, keep both if they exist
cols = [c for c in cols if c in events.columns]

# Biggest +30m moves
events.sort_values("ret_30m", ascending=False)[cols].head(10)

,user_name,symbol,dt0,p0,dt5,p5,dt30,p30,ret_5m,ret_30m,text
300,Joe Biden,SPY,2012-09-18 14:10:00-04:00,146.3700,2012-09-18 14:15:00-04:00,146.3900,2012-09-18 14:40:00-04:00,189.54,0.000137,0.294937,vp the president andhavedifferent way forward ...
301,Joe Biden,QQQ,2012-09-18 14:10:00-04:00,70.1699,2012-09-18 14:15:00-04:00,70.1800,2012-09-18 14:40:00-04:00,88.15,0.000144,0.256237,vp the president andhavedifferent way forward ...
511,Joe Biden,QQQ,2014-05-14 15:10:00-04:00,87.8000,2014-05-14 15:15:00-04:00,87.8400,2014-05-14 15:40:00-04:00,107.1772,0.000456,0.220697,we have to rebuild the infrastructure in this ...
56,Joe Biden,SPY,2020-12-23 15:30:00-05:00,368.7700,2020-12-23 15:35:00-05:00,368.8099,2020-12-23 16:00:00-05:00,435.33,0.000108,0.180492,this cybersecurity attack happened on donald t...
690,Joe Biden,SPY,2020-03-23 14:12:00-04:00,221.8300,2020-03-23 14:17:00-04:00,222.6400,2020-03-23 14:42:00-04:00,259.31,0.003651,0.168958,president trump and mitch mcconnell are trying...
57,Joe Biden,QQQ,2020-12-23 15:30:00-05:00,309.3000,2020-12-23 15:35:00-05:00,309.5200,2020-12-23 16:00:00-05:00,361.09,0.000711,0.167443,this cybersecurity attack happened on donald t...
53,Elon Musk,QQQ,2023-01-06 09:47:24-05:00,261.0399,2023-01-06 09:52:24-05:00,261.3327,2023-01-06 10:17:24-05:00,301.17,0.001122,0.153732,twitter has at least 10 million wokeys
275,Donald Trump,QQQ,2016-06-08 12:14:32-04:00,110.4400,2016-06-08 12:19:32-04:00,110.4200,2016-06-08 12:44:32-04:00,125.45,-0.000181,0.135911,historic whathistoric is our national debt rec...
691,Joe Biden,QQQ,2020-03-23 14:12:00-04:00,169.0300,2020-03-23 14:17:00-04:00,169.6500,2020-03-23 14:42:00-04:00,190.592,0.003668,0.127563,president trump and mitch mcconnell are trying...
484,Joe Biden,SPY,2020-04-02 14:59:00-04:00,248.7800,2020-04-02 15:04:00-04:00,249.0450,2020-04-02 15:29:00-04:00,279.78,0.001065,0.124608,the economic damage from this public health cr...


In [ ]:
events.sort_values("ret_30m", ascending=True)[cols].head(10)

,user_name,symbol,dt0,p0,dt5,p5,dt30,p30,ret_5m,ret_30m,text
379,Joe Biden,QQQ,2022-03-29 14:33:00-04:00,369.8050,2022-03-29 14:38:00-04:00,370.2700,2022-03-29 15:03:00-04:00,294.55,0.001257,-0.203499,my budget makes the investments needed to redu...
378,Joe Biden,SPY,2022-03-29 14:33:00-04:00,459.8699,2022-03-29 14:38:00-04:00,460.3800,2022-03-29 15:03:00-04:00,388.375,0.001109,-0.155468,my budget makes the investments needed to redu...
149,Joe Biden,QQQ,2021-11-17 14:49:01-05:00,398.1200,2021-11-17 14:54:01-05:00,398.0768,2021-11-17 15:19:01-05:00,342.29,-0.000109,-0.140234,because of the bipartisan infrastructure law n...
602,Joe Biden,SPY,2020-02-21 14:59:00-05:00,333.2500,2020-02-21 15:04:00-05:00,333.1089,2020-02-21 15:29:00-05:00,291.6,-0.000423,-0.124981,donald trump is the most corrupt president in ...
603,Joe Biden,QQQ,2020-02-21 14:59:00-05:00,230.0300,2020-02-21 15:04:00-05:00,229.9300,2020-02-21 15:29:00-05:00,203.17,-0.000435,-0.116767,donald trump is the most corrupt president in ...
595,Joe Biden,QQQ,2022-08-16 14:10:01-04:00,334.1699,2022-08-16 14:15:01-04:00,334.2603,2022-08-16 14:40:01-04:00,296.77,0.000271,-0.111919,democrats sided with the american people with ...
594,Joe Biden,SPY,2022-08-16 14:10:01-04:00,431.4700,2022-08-16 14:15:01-04:00,431.5280,2022-08-16 14:40:01-04:00,393.73,0.000134,-0.087468,democrats sided with the american people with ...
148,Joe Biden,SPY,2021-11-17 14:49:01-05:00,468.3900,2021-11-17 14:54:01-05:00,468.3100,2021-11-17 15:19:01-05:00,430.26,-0.000171,-0.081407,because of the bipartisan infrastructure law n...
480,Joe Biden,SPY,2020-03-19 14:30:00-04:00,244.2000,2020-03-19 14:35:00-04:00,244.9600,2020-03-19 15:00:00-04:00,224.85,0.003112,-0.079238,the obama biden administration set up the whit...
119,Donald Trump,QQQ,2015-12-01 15:20:40-05:00,114.8690,2015-12-01 15:25:40-05:00,114.9000,2015-12-01 15:50:40-05:00,105.93,0.000270,-0.077819,look at the editorialwas just sent from the ny...


In [ ]:
events["user_name"].value_counts()

,count
user_name,
Donald Trump,300
Joe Biden,256
Elon Musk,178


In [ ]:
events["vol_5m_abs"]  = events["ret_5m"].abs()
events["vol_30m_abs"] = events["ret_30m"].abs()

events["vol_5m_sq"]   = events["ret_5m"]**2
events["vol_30m_sq"]  = events["ret_30m"]**2

In [ ]:
events.groupby(["user_name","symbol"]).agg(
    n=("ret_30m","count"),
    mean_ret_5m=("ret_5m","mean"),
    mean_ret_30m=("ret_30m","mean"),
    mean_vol5=("vol_5m_abs","mean"),
    mean_vol30=("vol_30m_abs","mean"),
).sort_values("n", ascending=False)

n  mean_ret_5m  mean_ret_30m  mean_vol5  mean_vol30
user_name    symbol                                                       
Donald Trump QQQ     150     0.000017      0.005052   0.000459    0.014519
             SPY     150     0.000011      0.002801   0.000367    0.010208
Joe Biden    SPY     128     0.000213      0.005394   0.000527    0.023574
             QQQ     128     0.000201      0.006546   0.000636    0.027329
Elon Musk    SPY      88     0.000114      0.002278   0.000503    0.007789
             QQQ      88     0.000079      0.003995   0.000712    0.011675

In [ ]:
events.to_csv("/content/events_returns_volatility_final.csv", index=False)
print("Saved CSV to /content/events_returns_volatility_final.csv")

Saved CSV to /content/events_returns_volatility_final.csv
